**Objetivo:** preparar as características (`X`), os rótulos (`y`), os grupos (`SITE_ID`) e os folds externos para os modelos de TEA no ABIDE I.

**Entradas:** `connectivity_matrices_abide_validado.pkl`, produzido pelo Notebook 05, e `phenotypic_abide.csv`.

**Processamento:** o Notebook 04 produz correlações de Pearson (`r`). Este notebook extrai as 19.900 conexões únicas, aplica Fisher z uma vez, recodifica os rótulos para `0 = Controle` e `1 = TEA` e define cinco folds agrupados por `SITE_ID`.

**Saída:** `Data/processed/dataset_abide_processado.pkl`, utilizado pelos Notebooks 08, 09 e 10.

**Padronização:** `StandardScaler` não é aplicado aqui; seus parâmetros devem ser ajustados somente com os dados de treino de cada fold nos notebooks de modelagem.

In [1]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code/Jupyter)
# ------------------------------------------------------------
import sys
from pathlib import Path
import random
import numpy as np

_cwd = Path.cwd().resolve()
for _cand in (_cwd, *_cwd.parents):
    if (_cand / "src" / "config.py").is_file():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        "Não foi possível localizar src/config.py a partir de "
        f"{_cwd}. Abra a pasta raiz do projeto no VS Code."
    )

if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()
random.seed(SEED)
np.random.seed(SEED)

print("Diretório atual:", _cwd)
print("Raiz do projeto:", _raiz)
print("BASE_DIR:", BASE_DIR)
print("Pasta raw:", PATHS["raw"])
print("Pasta conectividade:", PATHS["conectivity"])
print("Pasta processados:", PATHS["processed"])


Diretório atual: /home/maria/TCC_TEA
Raiz do projeto: /home/maria/TCC_TEA
BASE_DIR: /home/maria/TCC_TEA
Pasta raw: /home/maria/TCC_TEA/Data/raw
Pasta conectividade: /home/maria/TCC_TEA/Data/conectivity
Pasta processados: /home/maria/TCC_TEA/Data/processed


In [2]:
# ============================================================
# CARREGAMENTO E PREPARAÇÃO DAS CARACTERÍSTICAS
# ============================================================

import pickle
import pandas as pd

conn_path = Path(PATHS["conectivity"]) / "connectivity_matrices_abide_validado.pkl"
phen_path = Path(PATHS["raw"]) / "phenotypic_abide.csv"

for nome, caminho in [
    ("Matrizes validadas (Notebook 05)", conn_path),
    ("Dados fenotípicos", phen_path),
]:
    print(f"{nome}: {caminho} | existe={caminho.is_file()}")
    if not caminho.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}\n"
            "Confira src/config.py e execute os notebooks anteriores necessários."
        )

with open(conn_path, "rb") as f:
    conn_dict_original = pickle.load(f)

if not isinstance(conn_dict_original, dict) or len(conn_dict_original) == 0:
    raise ValueError("O arquivo de conectividade não contém um dicionário válido e não vazio.")

# Normalizar IDs e impedir colisões ao converter para texto.
conn_dict = {}
for sub_id, mat in conn_dict_original.items():
    sid = str(sub_id).strip()
    if sid in conn_dict:
        raise ValueError(f"ID duplicado após normalização para texto: {sid}")
    conn_dict[sid] = np.asarray(mat)

phenotypic = pd.read_csv(phen_path)
colunas_obrigatorias = {"SUB_ID", "DX_GROUP", "SITE_ID"}
faltantes = colunas_obrigatorias - set(phenotypic.columns)
if faltantes:
    raise ValueError(f"Colunas ausentes no CSV fenotípico: {sorted(faltantes)}")

phenotypic["SUB_ID"] = phenotypic["SUB_ID"].astype(str).str.strip()
if phenotypic["SUB_ID"].duplicated().any():
    exemplos = phenotypic.loc[phenotypic["SUB_ID"].duplicated(), "SUB_ID"].unique()[:10]
    raise ValueError(f"SUB_ID duplicado no CSV fenotípico. Exemplos: {exemplos.tolist()}")

phenotypic = phenotypic.set_index("SUB_ID", drop=False)

# Preservar a mesma ordem de IDs em X, y, groups e folds.
sub_ids = np.asarray(list(conn_dict.keys()), dtype=str)

ids_sem_fenotipo = [sid for sid in sub_ids if sid not in phenotypic.index]
if ids_sem_fenotipo:
    raise ValueError(
        f"{len(ids_sem_fenotipo)} matrizes não têm participante correspondente no CSV. "
        f"Exemplos: {ids_sem_fenotipo[:10]}"
    )

matrices = np.stack([conn_dict[sid] for sid in sub_ids])

if matrices.ndim != 3 or matrices.shape[1:] != (200, 200):
    raise ValueError(
        f"Dimensão inesperada das matrizes: {matrices.shape}; esperado (N, 200, 200)."
    )

if not np.issubdtype(matrices.dtype, np.number):
    raise TypeError(f"As matrizes precisam ser numéricas. Tipo encontrado: {matrices.dtype}")

if not np.isfinite(matrices).all():
    raise ValueError(
        f"As matrizes contêm valores não finitos: "
        f"NaN={int(np.isnan(matrices).sum())}; "
        f"Inf={int(np.isinf(matrices).sum())}."
    )

# Vetorização: triângulo superior sem diagonal, por linha.
# CC200: 200*199/2 = 19.900 conexões únicas.
roi_i, roi_j = np.triu_indices(200, k=1)
X = matrices[:, roi_i, roi_j].astype(np.float64, copy=False)

# O Notebook 04 produz correlações Pearson r, não Fisher z.
# Verificar o intervalo e aplicar Fisher z uma única vez; o clipping evita infinito em r=+/-1.
if np.any(np.abs(X) > 1.0 + 1e-7):
    raise ValueError(
        "As matrizes não parecem conter correlações Pearson em [-1, 1]. "
        "Confira a origem dos dados antes de aplicar Fisher z."
    )
EPSILON_R = 1e-7
X = np.arctanh(np.clip(X, -1.0 + EPSILON_R, 1.0 - EPSILON_R))

# Metadados na mesma ordem das matrizes.
y_original = pd.to_numeric(
    phenotypic.loc[sub_ids, "DX_GROUP"], errors="coerce"
).to_numpy()
groups = phenotypic.loc[sub_ids, "SITE_ID"].astype("string").str.strip().to_numpy()

if pd.isna(y_original).any():
    raise ValueError("DX_GROUP contém valores ausentes ou não numéricos.")
if pd.isna(groups).any() or np.any(groups == ""):
    raise ValueError("SITE_ID contém valores ausentes ou vazios.")
if not set(np.unique(y_original).tolist()).issubset({1, 2}):
    raise ValueError(
        "Valores inesperados em DX_GROUP: "
        f"{sorted(np.unique(y_original).tolist())}. Esperados 1=TEA e 2=Controle."
    )
if X.shape[1] != 19900:
    raise ValueError(f"Esperadas 19.900 características; encontradas {X.shape[1]}.")
if not (len(X) == len(y_original) == len(groups) == len(sub_ids)):
    raise ValueError("X, y_original, groups e sub_ids possuem tamanhos diferentes.")

print("=" * 50)
print("DADOS PREPARADOS")
print("=" * 50)
print("Matrizes:", matrices.shape)
print("X:", X.shape)
print("Participantes:", len(sub_ids))
print("IDs únicos:", len(np.unique(sub_ids)))
print("Sites únicos:", len(np.unique(groups)))
print("DX_GROUP original:", pd.Series(y_original).value_counts().sort_index().to_dict())
print("Intervalo Fisher z:", float(X.min()), "a", float(X.max()))
print("Ordem das características: triângulo superior, por linha")


Matrizes validadas (Notebook 05): /home/maria/TCC_TEA/Data/conectivity/connectivity_matrices_abide_validado.pkl | existe=True
Dados fenotípicos: /home/maria/TCC_TEA/Data/raw/phenotypic_abide.csv | existe=True
DADOS PREPARADOS
Matrizes: (871, 200, 200)
X: (871, 19900)
Participantes: 871
IDs únicos: 871
Sites únicos: 20
DX_GROUP original: {1: 403, 2: 468}
Intervalo Fisher z: -1.0973010393644558 a 1.9218875825738237
Ordem das características: triângulo superior, por linha


In [3]:
# ============================================================
# RECODIFICAÇÃO DOS RÓTULOS
# ============================================================
# ABIDE I: DX_GROUP 1 = TEA; DX_GROUP 2 = Controle
# Modelos: 0 = Controle; 1 = TEA

y_bin = np.where(y_original == 1, 1, 0).astype(np.int64)

if len(y_bin) != len(X):
    raise ValueError("O número de rótulos não corresponde ao número de participantes.")

print("=" * 50)
print("DISTRIBUIÇÃO DAS CLASSES")
print("=" * 50)
print("Controle (0):", int(np.sum(y_bin == 0)))
print("TEA (1):", int(np.sum(y_bin == 1)))
print("Distribuição [Controle, TEA]:", np.bincount(y_bin, minlength=2).tolist())
print("Total:", len(y_bin))


DISTRIBUIÇÃO DAS CLASSES
Controle (0): 468
TEA (1): 403
Distribuição [Controle, TEA]: [468, 403]
Total: 871


In [4]:
# ============================================================
# VALIDAÇÃO CRUZADA ESTRATIFICADA POR SITE_ID
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)
folds = list(sgkf.split(X, y_bin, groups))

print("=" * 50)
print("VALIDAÇÃO DOS FOLDS")
print("=" * 50)

contagem_teste = np.zeros(len(X), dtype=int)

for i, (train_idx, test_idx) in enumerate(folds):
    sites_treino = set(groups[train_idx])
    sites_teste = set(groups[test_idx])
    intersecao = sites_treino & sites_teste
    classes_treino = np.unique(y_bin[train_idx])
    classes_teste = np.unique(y_bin[test_idx])
    contagem_teste[test_idx] += 1

    print(
        f"Fold {i}: treino={len(train_idx)} | teste={len(test_idx)} | "
        f"classes_treino={classes_treino.tolist()} | "
        f"classes_teste={classes_teste.tolist()} | "
        f"sites_teste={len(sites_teste)}"
    )

    if intersecao:
        raise RuntimeError(
            f"Fold {i}: SITE_ID aparece em treino e teste: {sorted(intersecao)}"
        )
    if len(classes_treino) < 2:
        raise RuntimeError(f"Fold {i}: treino não contém as duas classes.")
    if len(classes_teste) < 2:
        raise RuntimeError(
            f"Fold {i}: teste não contém as duas classes. "
            "Não calcule AUC como se esse fold fosse válido; reveja a estratégia "
            "de validação por site junto com os notebooks de modelagem."
        )

if not np.all(contagem_teste == 1):
    raise RuntimeError("Cada participante deve aparecer no teste exatamente uma vez.")

print("OK: sites separados entre treino/teste; cada participante aparece uma vez no teste.")


VALIDAÇÃO DOS FOLDS
Fold 0: treino=830 | teste=41 | classes_treino=[0, 1] | classes_teste=[0, 1] | sites_teste=1
Fold 1: treino=698 | teste=173 | classes_treino=[0, 1] | classes_teste=[0, 1] | sites_teste=4
Fold 2: treino=706 | teste=165 | classes_treino=[0, 1] | classes_teste=[0, 1] | sites_teste=5
Fold 3: treino=547 | teste=324 | classes_treino=[0, 1] | classes_teste=[0, 1] | sites_teste=6
Fold 4: treino=703 | teste=168 | classes_treino=[0, 1] | classes_teste=[0, 1] | sites_teste=4
OK: sites separados entre treino/teste; cada participante aparece uma vez no teste.


In [5]:
# ============================================================
# SALVAR DATASET PROCESSADO PARA OS NOTEBOOKS 08, 09 E 10
# ============================================================
# Não aplicar StandardScaler aqui: qualquer transformação que aprenda
# parâmetros deve ser ajustada somente no treino de cada fold.
# Fisher z já foi aplicada em X: o Notebook 04 gera correlações Pearson r.

dataset_processado = {
    "X": X,
    "y": y_bin,
    "groups": groups,
    "sub_ids": sub_ids,
    "folds": folds,
    "n_splits": N_SPLITS,
    "seed": SEED,
    "atlas": "CC200",
    "n_rois": 200,
    "n_features": 19900,
    "ordem_atributos": "triu_linha",
    "fisher_z": True,
    "transformacao_fisher_z_aplicada_no_notebook_07": True,
}

out_path = Path(PATHS["processed"]) / "dataset_abide_processado.pkl"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "wb") as f:
    pickle.dump(dataset_processado, f, protocol=pickle.HIGHEST_PROTOCOL)

print("=" * 50)
print("DATASET PROCESSADO SALVO")
print("=" * 50)
print("Arquivo:", out_path)
print("X:", X.shape)
print("y:", y_bin.shape)
print("groups:", groups.shape)
print("Folds:", len(folds))
print("Seed:", SEED)
print("Ordem dos atributos:", dataset_processado["ordem_atributos"])
print("Fisher z aplicada:", dataset_processado["fisher_z"])
print("Padronização aprendida: ajustar somente no treino de cada fold do notebook de modelagem.")


DATASET PROCESSADO SALVO
Arquivo: /home/maria/TCC_TEA/Data/processed/dataset_abide_processado.pkl
X: (871, 19900)
y: (871,)
groups: (871,)
Folds: 5
Seed: 42
Ordem dos atributos: triu_linha
Fisher z aplicada: True
Padronização aprendida: ajustar somente no treino de cada fold do notebook de modelagem.
